# Proyecto Final Integrador
## Diplomado en Ciencia de Datos y Análisis Avanzado — Unidad N°10

**Tema:** Predicción de mora en préstamos personales

**Dataset:** Credit Risk Dataset (Kaggle, `laotse/credit-risk-dataset`)




##1. Comprensión del negocio
1.1 Contexto y problema
Una entidad financiera (banco o fintech) otorga préstamos personales y asume una pérdida cuando el cliente cae en mora. Buscamos estimar, durante la evaluación inicial de cada solicitud, la probabilidad de mora (loan_status = 1) para priorizar revisiones y ajustar las condiciones de aprobación (monto, tasa, garantías) según el perfil de riesgo.

1.2 Momento de predicción
Definimos como momento de predicción la evaluación inicial de la solicitud, previa a la asignación de las condiciones definitivas (calificación loan_grade y tasa loan_int_rate). Esta definición determina qué variables son admisibles como predictoras y fundamenta la exclusión de dos variables por posible data leakage (sección 3.2).

1.3 Objetivo medible y KPI
Objetivo: construir un clasificador binario que estime P(mora) a partir de las variables disponibles en la evaluación inicial.
KPI primario: Recall de la clase mora ≥ 85 % sobre el conjunto de test, con la mayor Precision compatible con ese Recall.
KPI secundarios: PR-AUC, exposición económica en mora detectada y valor económico neto bajo escenarios de pérdida (sección 6).
Benchmark de negocio: política de aprobar todas las solicitudes (Recall = 0 %, PR-AUC igual a la prevalencia de mora).
Reglas fijadas antes de entrenar (ex ante). (i) El KPI es Recall ≥ 85 % en la clase mora. (ii) El umbral de decisión se elige solo con probabilidades out-of-fold de entrenamiento. (iii) El modelo final se elige por PR-AUC en validación cruzada. (iv) El conjunto de test no interviene en ninguna decisión y se reportan los resultados aunque sean adversos.

1.4 Justificación de las métricas
Con una prevalencia de mora cercana al 22 %, un clasificador que predice siempre "no mora" alcanza un accuracy del 78 % sin detectar ningún moroso; por eso descartamos el accuracy como criterio. Los costos de error son asimétricos: un falso negativo implica otorgar un crédito que no se recupera; un falso positivo implica resignar el margen de un buen cliente. Priorizamos el Recall de la clase positiva y reportamos Precision, F1, ROC-AUC y PR-AUC, que es más informativa que la ROC-AUC bajo desbalance (Saito & Rehmsmeier, 2015).

1.5 Hipótesis de trabajo
H1: variables observables en la solicitud (loan_percent_income, person_income, person_home_ownership, cb_person_default_on_file) se asocian con la mora.
H2: modelos no lineales basados en árboles superan a un modelo lineal en PR-AUC, por interacciones entre ingreso, monto y situación habitacional.
1.6 Restricciones
Dataset estático (no longitudinal) y sin variables de fecha; no captura cambios posteriores a la aprobación ni permite detectar mezcla de ciclos económicos. Es un dataset público con fines educativos, por lo que no representa la política de riesgo de una entidad específica. No contiene información de recuperos posteriores a la mora.

##2.Comprensión de los Datos

2.1 Entorno y carga


In [ ]:
import os, sys, time, platform, warnings, subprocess
try: import xgboost except ImportError: # p. ej. entorno nuevo subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost'], check=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

SEED = 42 COLORES = ['#4C72B0', '#C44E52'] # 0 = no mora, 1 = mora

Carpeta de salidas: tablas (CSV) y figuras que se usan en el informe

In [ ]:
os.makedirs('results', exist_ok=True)
def guardar_fig(nombre):
  plt.savefig(f'results/{nombre}.png', dpi=150, bbox_inches='tight')

In [ ]:
# Carga: local (por defecto) o Google Drive si se ejecuta en Colab sin el archivo local
RUTA = next((r for r in ['data/credit_risk_dataset.csv', 'credit_risk_dataset.csv'] if os.path.exists(r)), None)
if RUTA is None:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        RUTA = '/content/drive/MyDrive/credit_risk_dataset.csv'
    except ImportError:
        raise FileNotFoundError('Colocar credit_risk_dataset.csv en el directorio del notebook.')

df = pd.read_csv(RUTA)
print(f'Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}')
df.head()

In [ ]:
df.info()

### 2.2 Diccionario de variables

| Variable | Descripción | Tipo | Rol |
|---|---|---|---|
| `person_age` | Edad del solicitante (años) | numérica entera | Predictora |
| `person_income` | Ingreso anual | numérica entera | Predictora |
| `person_home_ownership` | Situación habitacional (RENT, OWN, MORTGAGE, OTHER) | categórica | Predictora |
| `person_emp_length` | Antigüedad laboral (años) | numérica continua | Predictora |
| `loan_intent` | Destino del préstamo (6 categorías) | categórica | Predictora |
| `loan_grade` | Calificación de riesgo asignada (A–G) | categórica ordinal | **Excluida (posible leakage)** |
| `loan_amnt` | Monto solicitado | numérica entera | Predictora |
| `loan_int_rate` | Tasa de interés asignada (%) | numérica continua | **Excluida (posible leakage)** |
| `loan_percent_income` | Monto / ingreso anual | numérica continua | Predictora |
| `cb_person_default_on_file` | Incumplimiento previo en buró (Y/N) | categórica binaria | Predictora |
| `cb_person_cred_hist_length` | Antigüedad del historial crediticio (años) | numérica entera | Predictora |
| `loan_status` | **Objetivo:** 1 = mora, 0 = pagó | binaria | Objetivo |

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

2.3 Variable objetivo

In [ ]:
conteo = df['loan_status'].value_counts().sort_index()
print(pd.DataFrame({'registros': conteo, '%': (conteo / len(df) * 100).round(2)}))

fig, ax = plt.subplots(figsize=(5, 3.6))
sns.countplot(x='loan_status', hue='loan_status', data=df, palette=COLORES, legend=False, ax=ax)
ax.set(title='Distribución de loan_status', xlabel='0 = no mora | 1 = mora', ylabel='Registros')
guardar_fig('fig_target'); plt.show()

La prevalencia de mora es del 21,8 % (7.108 de 32.581 registros): desbalance moderado. Lo abordamos con (i) partición estratificada, (ii) métricas sensibles a la clase minoritaria y (iii) selección explícita del umbral de decisión (sección 4).

2.4 Valores nulos

In [ ]:
nulos = df.isna().sum()
tabla_nulos = pd.DataFrame({'Nulos': nulos, '% Nulos': (nulos / len(df) * 100).round(2)})
tabla_nulos[tabla_nulos['Nulos'] > 0].to_csv('results/eda_nulos.csv')
tabla_nulos[tabla_nulos['Nulos'] > 0]

Solo person_emp_length (895 registros, 2,75 %) y loan_int_rate (3.116 registros, 9,56 %) presentan nulos. Ninguna proporción justifica descartar la variable. Para person_emp_length imputaremos con la mediana de entrenamiento dentro de un pipeline (sección 3.4). loan_int_rate queda fuera del modelado por la decisión de la sección 3.2.

2.5 Duplicados

In [ ]:
print('Filas totalmente duplicadas:', df.duplicated().sum())
df[df.duplicated()].head()

Detectamos 165 filas idénticas en todas las variables. Sin un identificador de cliente no podemos confirmar que sean el mismo individuo, pero la coincidencia exacta en las 12 columnas, varias de ellas continuas, hace improbable que sean casos independientes. Las eliminaremos para evitar sobreponderar observaciones y reducir el riesgo de que una misma fila caiga en entrenamiento y test.

2.6 Outliers e inconsistencias

a) person_age

In [ ]:
print('Registros con edad > 80:', (df['person_age'] > 80).sum())
df.loc[df['person_age'] > 80, ['person_age', 'person_emp_length', 'person_income', 'loan_status']]

Hay 7 registros con edad > 80. Cinco de ellos (123 y 144 años) son imposibles y los tratamos como errores de carga. Los de 84 y 94 años son atípicos pero plausibles y los conservamos.

b) person_emp_length

In [ ]:
# Criterio de coherencia: antigüedad laboral <= edad - 14 (edad mínima de ingreso al empleo)
incons = df[df['person_emp_length'] > (df['person_age'] - 14)]
print('Antigüedad laboral inconsistente con la edad:', len(incons))
incons[['person_age', 'person_emp_length', 'person_income']]

Dos registros declaran 123 años de antigüedad con 21 y 22 años de edad. Son errores de carga evidentes.

c) person_income

In [ ]:
print('Ingreso máximo:', f"{df['person_income'].max():,}")
print('Registros con ingreso > 1.000.000:', (df['person_income'] > 1_000_000).sum())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
sns.boxplot(x=df['person_income'], ax=axes[0], color='#9ecae1')
axes[0].set_title('person_income (escala original)')
sns.histplot(np.log1p(df['person_income']), bins=50, ax=axes[1], color='#9ecae1')
axes[1].set_title('log(1 + person_income)')
plt.tight_layout(); plt.show()

Nueve registros superan USD 1.000.000 (máximo USD 6.000.000). Un ingreso elevado no constituye por sí mismo un error, por lo que no los eliminamos. La distribución original tiene fuerte asimetría positiva; el logaritmo la simetriza (decisión de preparación: aplicar log1p a person_income en el pipeline, sección 3.4). Esta transformación reduce la influencia de valores extremos en modelos lineales y es neutra para árboles.

2.7 Relación de las variables con la mora

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
orden_grade = sorted(df['loan_grade'].unique())
for ax, col, orden in zip(axes.ravel(),
                          ['loan_grade', 'person_home_ownership', 'loan_intent', 'cb_person_default_on_file'],
                          [orden_grade, None, None, None]):
    tasa = df.groupby(col)['loan_status'].mean().mul(100)
    if orden is not None:
        tasa = tasa.reindex(orden)
    else:
        tasa = tasa.sort_values(ascending=False)
    sns.barplot(x=tasa.index, y=tasa.values, ax=ax, color='#C44E52')
    ax.axhline(df['loan_status'].mean() * 100, ls='--', c='gray', lw=1, label='Prevalencia global')
    ax.set(title=f'Tasa de mora por {col}', xlabel='', ylabel='% mora')
    ax.tick_params(axis='x', rotation=30)
axes[0, 0].legend()
plt.tight_layout(); guardar_fig('fig_eda_categoricas'); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, col in zip(axes, ['loan_percent_income', 'person_income', 'loan_amnt']):
    sns.boxplot(x='loan_status', y=col, hue='loan_status', data=df, palette=COLORES, legend=False, showfliers=False, ax=ax)
    ax.set_title(f'{col} según mora')
plt.tight_layout(); guardar_fig('fig_eda_numericas'); plt.show()

bins = [0, .10, .20, .30, .40, 1.0]
etiquetas = ['≤10 %', '10–20 %', '20–30 %', '30–40 %', '>40 %']
tramo = pd.cut(df['loan_percent_income'], bins=bins, labels=etiquetas)
tabla_tramo = df.groupby(tramo, observed=True)['loan_status'].agg(registros='size', mora_pct=lambda s: s.mean() * 100).round(1)
display(tabla_tramo)
tabla_tramo.to_csv('results/eda_mora_por_tramo_loan_percent_income.csv')

In [ ]:
cols_num = ['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate',
            'loan_percent_income', 'cb_person_cred_hist_length', 'loan_status']
corr = df[cols_num].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlación de Pearson (incluye loan_status)')
guardar_fig('fig_correlacion'); plt.show()

Conclusiones del análisis exploratorio

loan_percent_income es la variable más asociada con la mora (r = 0,38). Los morosos presentan una mediana de 0,24 frente a 0,13 en los no morosos. La relación no es gradual: la mora pasa de 21,9 % en el tramo 20–30 % a 68,7 % en 30–40 % y 74,2 % por encima de 40 %. Hay un cambio de régimen alrededor del 30 % del ingreso comprometido.
person_income se asocia negativamente (r = −0,14; mediana de 41.498 en morosos vs. 60.000 en no morosos) y loan_amnt positivamente (r = 0,11). Ambas se vinculan entre sí a través de loan_percent_income (r = 0,57 con loan_amnt), sin colinealidad severa.
Situación habitacional: mora de 31,6 % en RENT y 30,8 % en OTHER, frente a 12,6 % en MORTGAGE y 7,5 % en OWN.
Destino del préstamo: de 28,6 % (DEBTCONSOLIDATION) a 14,8 % (VENTURE).
Incumplimiento previo: 37,8 % de mora con antecedente vs. 18,4 % sin él.
person_age y cb_person_cred_hist_length casi no se asocian linealmente con la mora (|r| < 0,03).
Los resultados respaldan H1: las variables de capacidad de pago y situación habitacional discriminan la mora. La no linealidad observada en loan_percent_income anticipa una ventaja de los modelos basados en árboles (H2).

##3. Preparación de datos

###3.1 Limpieza

Aplicamos solo las decisiones que no requieren estimar parámetros a partir de los datos (así evitamos leakage desde test). Usamos reglas explícitas en lugar de valores puntuales:

1. Edad > 100 años → error de carga.
2. Antigüedad laboral > edad − 14 → inconsistencia lógica (los nulos se conservan para imputarlos después).
3. Filas duplicadas exactas.

In [ ]:
df_limpio = df.copy()
n0 = len(df_limpio)

mask_edad = df_limpio['person_age'] > 100
mask_emp  = df_limpio['person_emp_length'] > (df_limpio['person_age'] - 14)   # NaN -> False (se conserva)
df_limpio = df_limpio[~mask_edad & ~mask_emp]
n1 = len(df_limpio)

df_limpio = df_limpio.drop_duplicates()
n2 = len(df_limpio)

resumen = pd.DataFrame({
    'Paso': ['Dataset original', 'Edad > 100 o antigüedad inconsistente', 'Duplicados exactos', 'Dataset limpio'],
    'Filas eliminadas': [0, n0 - n1, n1 - n2, n0 - n2],
    'Filas restantes': [n0, n1, n2, n2]})
resumen.to_csv('results/limpieza_pasos.csv', index=False)
display(resumen)
print(f'Porcentaje eliminado: {(n0 - n2) / n0 * 100:.2f} %')
print('Nulos restantes:\n', df_limpio.isna().sum()[df_limpio.isna().sum() > 0].to_string())

Eliminamos 172 registros (0,53 %), por lo que conservamos prácticamente toda la información. La prevalencia de mora no se altera de forma apreciable.

###3.2 Evaluación de loan_grade y loan_int_rate por data leakage

Data leakage ocurre cuando una variable de entrenamiento contiene información que no estaría disponible en el momento real de la predicción o que codifica el resultado que se quiere predecir (Kaufman et al., 2012). Analizamos dos candidatas.

In [ ]:
print('Tasa de interés promedio y mediana según loan_grade:')
display(df_limpio.groupby('loan_grade')['loan_int_rate'].agg(['count', 'mean', 'median']).round(2))

print('Correlación de loan_int_rate con variables del préstamo y del solicitante:')
display(df_limpio[['loan_int_rate', 'loan_amnt', 'loan_percent_income', 'person_income']].corr().round(2))

print('Tasa promedio según destino del préstamo:')
display(df_limpio.groupby('loan_intent')['loan_int_rate'].agg(['mean', 'median']).round(2))

loan_int_rate crece de forma casi determinística con loan_grade (7,3 % en A a 20,3 % en G) y casi no varía con el destino, el monto o el ingreso: refleja la calificación de riesgo, no las características del préstamo. loan_grade es una calificación de riesgo ya asignada, y a la vez la variable más asociada con la mora (10 % en A, 98 % en G).

Como la predicción debe ocurrir antes de asignar esas condiciones, ambas variables incorporarían el resultado de una evaluación de riesgo previa y no estarían disponibles en el momento definido. Las excluimos del modelo principal. En la sección 5.6 cuantificamos el efecto de esta decisión con un experimento de sensibilidad.

###3.3 Variables predictoras y partición entrenamiento/test
Particionamos 80 % / 20 % de forma estratificada por loan_status para conservar la prevalencia en ambos conjuntos. El conjunto de test se mantiene aislado hasta la evaluación final: toda decisión de modelado (hiperparámetros, umbral, selección del modelo) se toma con validación cruzada sobre entrenamiento.

In [ ]:
from sklearn.model_selection import train_test_split

X = df_limpio.drop(columns=['loan_status', 'loan_grade', 'loan_int_rate'])
y = df_limpio['loan_status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED, stratify=y)

print('Entrenamiento:', X_train.shape, '| Test:', X_test.shape)
print(pd.DataFrame({'train %': y_train.value_counts(normalize=True) * 100,
                    'test %':  y_test.value_counts(normalize=True) * 100}).round(2))

###3.4 Preprocesamiento
Definimos un `ColumnTransformer` encapsulado en un `Pipeline` con el modelo. Así los parámetros de las transformaciones (mediana, media y desvío) se **aprenden solo con los datos de entrenamiento de cada partición** de la validación cruzada, evitando *leakage* de preprocesamiento.

- `person_income`: `log1p` + estandarización.
- Resto de numéricas: imputación por mediana (afecta solo a `person_emp_length`) + estandarización.
- Categóricas: *One-Hot Encoding* con `handle_unknown='ignore'`.

La estandarización es necesaria para la regresión logística (optimización y regularización sensibles a la escala) y neutra para los modelos de árboles.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

vars_log = ['person_income']
vars_num = ['person_age', 'person_emp_length', 'loan_amnt', 'loan_percent_income', 'cb_person_cred_hist_length']
vars_cat = ['person_home_ownership', 'loan_intent', 'cb_person_default_on_file']

def crear_preprocesador(v_log=vars_log, v_num=vars_num, v_cat=vars_cat):
    log1p = FunctionTransformer(np.log1p, feature_names_out='one-to-one')
    return ColumnTransformer([
        ('log', Pipeline([('log', log1p), ('esc', StandardScaler())]), v_log),
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('esc', StandardScaler())]), v_num),
        ('cat', OneHotEncoder(handle_unknown='ignore'), v_cat)])

crear_preprocesador()

In [ ]:
# Copia del dataset original
df_limpio = df.copy()
filas_iniciales = len(df_limpio)

# Eliminar edades claramente inconsistentes
df_limpio = df_limpio[~df_limpio['person_age'].isin([123, 144])]

# Eliminar antigüedades laborales incompatibles con la edad
df_limpio = df_limpio[df_limpio['person_emp_length'].isna() | (df_limpio['person_emp_length'] <= (df_limpio['person_age'] - 14))]

print("Filas iniciales:", filas_iniciales)
print("Filas después de eliminar inconsistencias:", len(df_limpio))


Se eliminaron los registros con edades de 123 y 144 años, considerados valores inconsistentes. Asimismo, se eliminaron los registros cuya antigüedad laboral resultaba incompatible con la edad del solicitante. Los valores nulos de person_emp_length se conservaron en esta etapa para ser tratados posteriormente mediante imputación.

**Eliminación de filas con valores idénticos**

In [ ]:
antes_duplicados = len(df_limpio)
df_limpio = df_limpio.drop_duplicates()

eliminados_duplicados = antes_duplicados - len(df_limpio)

print("Filas con valores idénticos eliminadas:", eliminados_duplicados)

Debido a que el dataset no dispone de un identificador único de cliente, no es posible confirmar que las filas con valores idénticos correspondan efectivamente al mismo individuo. Sin embargo, dada la coincidencia completa en todas las variables, se decidió eliminarlas para evitar dar un peso excesivo a determinadas observaciones durante el entrenamiento.

**Tratamiento de `person_income`**


Los valores extremos de `person_income` no se eliminaron, dado que un ingreso elevado no constituye por sí mismo un error de carga. Se conservarán las observaciones válidas y se evaluará posteriormente la conveniencia de aplicar una transformación logarítmica dentro del proceso de preparación para el modelado.



**Evaluación de `loan_grade` y `loan_int_rate`  por posible data leakage**


In [ ]:
#Analizar loan_int_rate respecto de las características del préstamo

# 1. Tasa de interés según destino del préstamo
tabla_intent = (df_limpio.groupby('loan_intent')['loan_int_rate'].agg(['count', 'mean', 'median']).round(2))
display(tabla_intent)

# 2. Relación entre tasa de interés, monto e ingreso
tabla_corr = (df_limpio[['loan_int_rate', 'loan_amnt', 'loan_percent_income', 'person_income']].corr().round(2))
display(tabla_corr)

# 3. Tasa de interés promedio según loan_grade
tabla_grade_tasa = (df_limpio.groupby('loan_grade')['loan_int_rate'].agg(['count', 'mean', 'median']).round(2))
display(tabla_grade_tasa)




 El análisis muestra una fuerte relación entre `loan_grade` y `loan_int_rate`: la tasa de interés promedio aumenta de manera sostenida a medida que empeora la calificación crediticia, desde 7,33% para la categoría A hasta 20,25% para la categoría G. En cambio, loan_int_rate presenta poca variación según el destino del préstamo y una baja correlación lineal con variables como el monto solicitado, la relación préstamo/ingreso y el ingreso del solicitante.

Considerando que el objetivo del proyecto es realizar la predicción durante la evaluación inicial de la solicitud, antes de la asignación de las condiciones definitivas del préstamo, se decidió no utilizar `loan_grade` ni `loan_int_rate` como variables predictoras del modelo principal. De esta manera, se busca evitar incorporar información que podría originarse en una instancia posterior del proceso de evaluación crediticia y generar data leakage.

In [ ]:
#Verificación final de la limpieza
filas_finales = len(df_limpio)
filas_eliminadas = filas_iniciales - filas_finales

print("Filas originales:", filas_iniciales)
print("Filas después de la limpieza:", filas_finales)
print("Filas eliminadas:", filas_eliminadas)
print(
    "Porcentaje eliminado:",
    round(filas_eliminadas / filas_iniciales * 100, 2),
    "%"
)

print("\nValores nulos restantes:")
print(df_limpio.isna().sum()[df_limpio.isna().sum() > 0])

Luego de aplicar los criterios de limpieza definidos, el dataset quedó conformado por 32.409 registros, habiéndose eliminado 172 observaciones (0,53% del total original). La baja proporción de registros eliminados permite conservar prácticamente la totalidad de la información disponible.

Luego de la limpieza permanecen valores nulos en `person_emp_length` (887 registros) y `loan_int_rate` (3.094 registros). Dado que `loan_int_rate` fue excluida del conjunto de variables predictoras por el posible riesgo de data leakage, sus valores faltantes no requieren imputación para el modelado. Los valores nulos de `person_emp_length` serán tratados posteriormente, una vez realizada la división entre los conjuntos de entrenamiento y test.

Tras la limpieza quedan dos registros con edad superior a 80 años (84 y 94 años). A diferencia de los valores de 123 y 144 años, son edades biológicamente posibles y no presentan inconsistencias con el resto de sus variables, por lo que se conservan en el dataset.

**4. Relevamiento de Fuentes de Datos**

**4.1 Definición de variables predictoras y objetivo**

In [ ]:
X = df_limpio.drop(columns=['loan_status', 'loan_grade', 'loan_int_rate'])
y = df_limpio['loan_status']

**4.2 División entrenamiento/test**

El conjunto de datos se divide en un 80% para entrenamiento y un 20% para test. La división se realiza de forma estratificada según la variable objetivo `loan_status`, con el fin de mantener aproximadamente la misma proporción de casos de mora y no mora en ambos conjuntos.

El conjunto de test se mantendrá separado y se utilizará únicamente para la evaluación final de los modelos.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print("Tamaño del conjunto de entrenamiento:", X_train.shape)
print("Tamaño del conjunto de test:", X_test.shape)

print("\nDistribución de loan_status en entrenamiento:")
print(y_train.value_counts(normalize=True).round(4) * 100)

print("\nDistribución de loan_status en test:")
print(y_test.value_counts(normalize=True).round(4) * 100)

La división estratificada permitió mantener prácticamente la misma distribución de la variable objetivo en ambos conjuntos. El conjunto de entrenamiento presenta un 21,87% de casos de mora, mientras que en el conjunto de test la proporción es del 21,88%. De esta manera, ambos conjuntos conservan el desbalance observado en el dataset original.

**4.3 Preprocesamiento**

Antes de entrenar los modelos es necesario preparar las variables predictoras. El preprocesamiento se define utilizando únicamente el conjunto de entrenamiento, evitando que información del conjunto de test intervenga en esta etapa.

Se realizará la imputación de los valores faltantes de `person_emp_length`, la codificación de las variables categóricas y el escalado de las variables numéricas. Estas transformaciones se integrarán en un mismo proceso para aplicarlas posteriormente de forma consistente durante el entrenamiento y la evaluación de los modelos  

**4.3.1 Clasificación de variables para el preprocesamiento**

A partir de los tipos de variables identificados durante el análisis exploratorio, se separan las variables numéricas y categóricas para aplicar posteriormente el tratamiento correspondiente a cada grupo.

In [ ]:
# Variables según su tipo para el preprocesamiento

variables_numericas = ['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_percent_income', 'cb_person_cred_hist_length']

variables_categoricas = ['person_home_ownership', 'loan_intent', 'cb_person_default_on_file']

**4.3.2 Definición del preprocesamiento**

Para preparar los datos para el modelado se aplicará un tratamiento diferenciado según el tipo de variable. En las variables numéricas, los valores faltantes de person_emp_length se imputarán utilizando la mediana del conjunto de entrenamiento y posteriormente se realizará un escalado. Las variables categóricas se transformarán mediante One-Hot Encoding.

Estas transformaciones se definirán dentro de un pipeline, de manera que los parámetros necesarios para el preprocesamiento se aprendan únicamente a partir de los datos de entrenamiento y luego se apliquen de forma consistente al conjunto de test.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Preprocesamiento de variables numéricas
pipeline_numerico = Pipeline(steps=[('imputacion', SimpleImputer(strategy='median')), ('escalado', StandardScaler())])

# Preprocesamiento de variables categóricas
pipeline_categorico = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))])

# Unificación del preprocesamiento
preprocesador = ColumnTransformer(transformers=[('num', pipeline_numerico, variables_numericas), ('cat', pipeline_categorico, variables_categoricas)])

**4.4 Modelado**

**4.4.1 Regresión Logística**

Como primer modelo se utilizará una Regresión Logística, adecuada para problemas de clasificación binaria. Este modelo se utilizará como una primera referencia para evaluar la capacidad predictiva de las variables seleccionadas y posteriormente comparar sus resultados con modelos de mayor complejidad.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Pipeline: preprocesamiento + modelo
modelo_logistico = Pipeline(steps=[('preprocesamiento', preprocesador), ('modelo', LogisticRegression(max_iter=1000, random_state=42))])

# Entrenamiento
modelo_logistico.fit(X_train, y_train)

**4.4.2 Evaluación inicial de la Regresión Logística**

In [ ]:
#1. Predicción
# Predicciones sobre el conjunto de test
y_pred_log = modelo_logistico.predict(X_test)

# Probabilidad estimada de mora
y_prob_log = modelo_logistico.predict_proba(X_test)[:, 1]

#2. Calcular las métricas
from sklearn.metrics import (confusion_matrix, classification_report, roc_auc_score, average_precision_score)

print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred_log))

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_log, digits=3))

print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_log), 3))
print("PR-AUC:", round(average_precision_score(y_test, y_prob_log), 3))

Con el umbral de clasificación estándar de 0,50, la Regresión Logística alcanza un ROC-AUC de 0,805 y un PR-AUC de 0,622. Para la clase de mora (1), obtiene una Precision de 0,737, un Recall de 0,404 y un F1-score de 0,522.

La matriz de confusión muestra que el modelo identifica correctamente 573 de los 1.418 casos de mora, mientras que 845 casos de mora son clasificados incorrectamente como no morosos (falsos negativos). Por lo tanto, si bien el modelo presenta una capacidad de discriminación razonable, con el umbral estándar de 0,50 no alcanza el objetivo de negocio definido de lograr un Recall mínimo del 85% para la clase de mora.

**4.4.3 Análisis del umbral de decisión**

Dado que con el umbral estándar de 0,50 el modelo alcanza un Recall de 40,4% para la clase de mora, por debajo del objetivo establecido del 85%, se analiza la posibilidad de modificar el punto de corte.

Para evitar utilizar el conjunto de test en la elección del umbral, se obtienen probabilidades sobre el conjunto de entrenamiento mediante validación cruzada estratificada de 5 particiones. A partir de estas probabilidades se analiza la relación entre Precision y Recall y se identifica el umbral que permite alcanzar un Recall mínimo del 85%.

In [ ]:
#Obtener probabilidades mediante validación cruzada
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import precision_recall_curve

# Validación cruzada estratificada de 5 particiones
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Probabilidades obtenidas mediante validación cruzada
y_prob_cv = cross_val_predict(modelo_logistico, X_train, y_train, cv=cv, method='predict_proba')[:, 1]

#Buscar el umbral compatible con Recall ≥ 85%
precision, recall, thresholds = precision_recall_curve(y_train, y_prob_cv)

# precision y recall tienen un elemento más que thresholds
precision_umbral = precision[:-1]
recall_umbral = recall[:-1]

# Umbrales que cumplen Recall >= 85%
cumple_objetivo = recall_umbral >= 0.85

# Elegimos, entre ellos, el que consigue mayor Precision
indice_mejor = np.argmax(np.where(cumple_objetivo, precision_umbral, -1))

umbral_85 = thresholds[indice_mejor]

print("Umbral seleccionado:", round(umbral_85, 3))
print("Recall:", round(recall_umbral[indice_mejor], 3))
print("Precision:", round(precision_umbral[indice_mejor], 3))

#Gráfico Precision-Recall
plt.figure(figsize=(7,5))

plt.plot(recall, precision)

plt.scatter(recall_umbral[indice_mejor], precision_umbral[indice_mejor], color='red', label=f'Umbral = {umbral_85:.3f}')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Curva Precision-Recall - Regresión Logística')
plt.legend()
plt.grid()
plt.show()

Mediante validación cruzada sobre el conjunto de entrenamiento, se identificó un umbral de aproximadamente 0,117 para alcanzar el objetivo de un Recall mínimo del 85%. Con este punto de corte, el Recall alcanza el 85,0%, mientras que la Precision se sitúa en 32,8%.

El resultado evidencia el compromiso existente entre ambas métricas: reducir el umbral permite identificar una mayor proporción de clientes que efectivamente entrarán en mora y disminuir los falsos negativos, pero al mismo tiempo aumenta la cantidad de clientes sin mora clasificados como riesgosos. Por este motivo, el cumplimiento del objetivo de Recall deberá evaluarse conjuntamente con la Precision y su impacto en el negocio.

**4.4.4 Evaluación de la Regresión Logística con el umbral seleccionado**

Una vez seleccionado el umbral mediante validación cruzada sobre el conjunto de entrenamiento, se evalúa su desempeño sobre el conjunto de test. De esta manera se analiza si el nivel de Recall buscado se mantiene sobre datos no utilizados para seleccionar el punto de corte.

In [ ]:
# Aplicación del umbral seleccionado al conjunto de test
y_pred_log_85 = (y_prob_log >= umbral_85).astype(int)

print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred_log_85))

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_log_85, digits=3))

print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_log), 3))
print("PR-AUC:", round(average_precision_score(y_test, y_prob_log), 3))

Con el umbral seleccionado de 0,117, la Regresión Logística alcanza en el conjunto de test un Recall de 84,9% para la clase de mora, prácticamente coincidente con el objetivo establecido del 85%. De los 1.418 casos de mora, el modelo identifica correctamente 1.204, mientras que 214 quedan sin detectar como falsos negativos.

Este aumento del Recall tiene como contrapartida una disminución de la Precision a 33,5%, debido al incremento de falsos positivos: 2.391 clientes sin mora son clasificados como riesgosos. Por lo tanto, el ajuste del umbral permite reducir considerablemente los casos de mora no detectados, aunque a costa de clasificar como riesgosos a un mayor número de clientes que finalmente no presentan mora.

El modelo mantiene un ROC-AUC de 0,805 y un PR-AUC de 0,622, ya que estas métricas se calculan a partir de las probabilidades estimadas y no dependen del umbral de clasificación elegido.

**4.5 Random Forest**

**4.5.1 Entrenamiento del modelo**

Como segundo algoritmo se utilizará un Random Forest, un modelo basado en múltiples árboles de decisión que permite captar relaciones no lineales e interacciones entre las variables. Su desempeño se comparará posteriormente con el obtenido mediante Regresión Logística, utilizando los mismos conjuntos de entrenamiento y test y las mismas métricas de evaluación.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Pipeline: preprocesamiento + modelo
modelo_rf = Pipeline(steps=[('preprocesamiento', preprocesador), ('modelo', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1))])

# Entrenamiento
modelo_rf.fit(X_train, y_train)

**4.5.2 Evaluación inicial del Random Forest**

Una vez entrenado el modelo Random Forest, se evalúa inicialmente su desempeño sobre el conjunto de test utilizando el umbral estándar de clasificación de 0,50. Esto permite obtener una primera medida de su capacidad predictiva y compararla posteriormente con la Regresión Logística.

In [ ]:
# Predicciones sobre el conjunto de test
y_pred_rf = modelo_rf.predict(X_test)

# Probabilidad estimada de mora
y_prob_rf = modelo_rf.predict_proba(X_test)[:, 1]

print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_rf, digits=3))

print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_rf), 3))
print("PR-AUC:", round(average_precision_score(y_test, y_prob_rf), 3))

Con el umbral estándar de 0,50, Random Forest obtiene un ROC-AUC de 0,877 y un PR-AUC de 0,785, superiores a los obtenidos con la Regresión Logística. Para la clase de mora alcanza una Precision de 89,2%, un Recall de 54,5% y un F1-score de 0,677.

De los 1.418 casos de mora, el modelo identifica correctamente 773, mientras que 645 quedan sin detectar como falsos negativos. Aunque presenta una mejora respecto de la Regresión Logística con el umbral estándar, el Recall continúa por debajo del objetivo mínimo del 85%

**4.5.3 Análisis del umbral de decisión**

Dado que el Recall obtenido con el umbral estándar continúa por debajo del objetivo establecido, se analiza un punto de corte alternativo. Para evitar utilizar el conjunto de test en la selección del umbral, se obtienen probabilidades mediante validación cruzada estratificada sobre el conjunto de entrenamiento. Entre los umbrales que permiten alcanzar un Recall mínimo del 85%, se selecciona aquel que presenta la mayor Precision.

In [ ]:
# Probabilidades mediante validación cruzada
y_prob_cv_rf = cross_val_predict(modelo_rf, X_train, y_train, cv=cv, method='predict_proba')[:, 1]

# Curva Precision-Recall
precision_rf, recall_rf, thresholds_rf = precision_recall_curve( y_train, y_prob_cv_rf)

precision_umbral_rf = precision_rf[:-1]
recall_umbral_rf = recall_rf[:-1]

# Umbrales que cumplen Recall >= 85%
cumple_objetivo_rf = recall_umbral_rf >= 0.85

# Entre ellos, seleccionar el de mayor Precision
indice_mejor_rf = np.argmax(np.where(cumple_objetivo_rf, precision_umbral_rf, -1))

umbral_85_rf = thresholds_rf[indice_mejor_rf]

print("Umbral seleccionado:", round(umbral_85_rf, 3))
print("Recall:", round(recall_umbral_rf[indice_mejor_rf], 3))
print("Precision:", round(precision_umbral_rf[indice_mejor_rf], 3))

#Gráfico Precision-Recall
plt.figure(figsize=(7,5))

plt.plot(recall_rf, precision_rf)

plt.scatter(recall_umbral_rf[indice_mejor_rf], precision_umbral_rf[indice_mejor_rf], color='red', label=f'Umbral = {umbral_85_rf:.3f}')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Curva Precision-Recall - Random Forest')
plt.legend()
plt.grid()
plt.show()

Mediante validación cruzada sobre el conjunto de entrenamiento, se identificó un umbral de aproximadamente 0,123 para alcanzar el objetivo de Recall mínimo del 85%. Con este punto de corte, Random Forest obtiene un Recall de 85,1% y una Precision de 39,6%.

Al igual que en la Regresión Logística, la reducción del umbral permite detectar una mayor proporción de clientes que posteriormente entran en mora, aunque aumenta la cantidad de falsos positivos. Sin embargo, para un nivel de Recall similar, Random Forest mantiene una Precision superior a la obtenida por la Regresión Logística (39,6% frente a 32,8%), lo que indica un mejor equilibrio entre la detección de clientes morosos y la clasificación incorrecta de clientes sin mora.

**4.5.4 Evaluación de Random Forest con el umbral seleccionado**

In [ ]:
# Aplicación del umbral seleccionado al conjunto de test
y_pred_rf_85 = (y_prob_rf >= umbral_85_rf).astype(int)

print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred_rf_85))

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_rf_85, digits=3))

print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_rf), 3))
print("PR-AUC:", round(average_precision_score(y_test, y_prob_rf), 3))

Con el umbral seleccionado de 0,123, Random Forest alcanza en el conjunto de test un Recall de 85,9% para la clase de mora, superando levemente el objetivo  establecido del 85%. De los 1.418 casos de mora, el modelo identifica correctamente 1.218, mientras que 200 quedan sin detectar como falsos negativos.

La Precision de la clase de mora es de 40,6%, lo que implica que el aumento del Recall se obtiene a costa de incrementar los falsos positivos: 1.781 clientes sin mora son clasificados como riesgosos.

El modelo obtiene además un ROC-AUC de 0,877 y un PR-AUC de 0,785. Estos resultados muestran que Random Forest presenta una mayor capacidad de discriminación que la Regresión Logística y, para un nivel de Recall cercano al objetivo de negocio, mantiene una Precision superior.

**4.6 Comparación de modelos**

Ambos modelos fueron evaluados sobre el mismo conjunto de test y utilizando las mismas métricas. Con el umbral ajustado para alcanzar un Recall cercano al 85%, Random Forest presenta un mejor desempeño general que la Regresión Logística.

La Regresión Logística alcanza un Recall de 84,9% y una Precision de 33,5%, mientras que Random Forest obtiene un Recall de 85,9% y una Precision de 40,6%. Esto implica que Random Forest logra detectar una proporción un tanto mayor de clientes con mora y, al mismo tiempo, reduce la cantidad de clientes sin mora clasificados incorrectamente como riesgosos.

Además, Random Forest obtiene mejores resultados en las métricas independientes del punto de corte, con un ROC-AUC de 0,877 frente a 0,805 y un PR-AUC de 0,785 frente a 0,622 de la Regresión Logística. En particular, la mejora en PR-AUC resulta relevante dado el desbalance existente en la variable objetivo.

En función de estos resultados, Random Forest se selecciona como el modelo de mejor desempeño entre los algoritmos evaluados, utilizando el umbral de decisión ajustado para priorizar la detección de clientes con riesgo de mora.

| Modelo              | Umbral | Recall mora | Precision mora |   F1 mora |   ROC-AUC |    PR-AUC |
| ------------------- | -----: | ----------: | -------------: | --------: | --------: | --------: |
| Regresión Logística |  0,117 |       84,9% |          33,5% |     0,480 |     0,805 |     0,622 |
| Random Forest       |  0,123 |   **85,9%** |      **40,6%** | **0,552** | **0,877** | **0,785** |


**4.7 Benchmark de negocio**

Como benchmark de negocio se considera un escenario base en el que se aprueban todas las solicitudes de crédito, sin utilizar un modelo predictivo de riesgo. En este escenario, todos los clientes que posteriormente entran en mora son aceptados y, por lo tanto, la entidad queda expuesta a la totalidad de esos casos.

El desempeño del modelo seleccionado se compara con este escenario para evaluar en qué medida permite identificar anticipadamente solicitudes asociadas a un mayor riesgo de mora.

In [ ]:
# Benchmark: aprobar a todos los solicitantes

morosos_test = (y_test == 1).sum()
morosos_detectados_rf = ((y_test == 1) & (y_pred_rf_85 == 1)).sum()
morosos_no_detectados_rf = ((y_test == 1) & (y_pred_rf_85 == 0)).sum()

print("Casos de mora en test:", morosos_test)
print("Morosos detectados por Random Forest:", morosos_detectados_rf)
print("Morosos no detectados:", morosos_no_detectados_rf)

print("Porcentaje de morosos detectados:", round(morosos_detectados_rf / morosos_test * 100, 2),"%")

En el escenario benchmark, en el que se aprueban todas las solicitudes sin utilizar un modelo predictivo, los 1.418 clientes que finalmente presentan mora no serían identificados previamente como clientes de mayor riesgo.

Con Random Forest y el umbral seleccionado, el modelo permite identificar anticipadamente 1.218 de esos casos, equivalentes al 85,9% del total de clientes que posteriormente entran en mora, mientras que 200 casos permanecen sin detectar.

Estos resultados muestran que el uso del modelo permite identificar una proporción elevada de las solicitudes asociadas posteriormente a mora. Sin embargo, la detección de un cliente riesgoso no implica necesariamente rechazar la solicitud ni evitar completamente una pérdida, sino que podría utilizarse como información adicional para aplicar distintas estrategias de gestión del riesgo

**4.8 Interpretación del modelo seleccionado**

Una vez seleccionado Random Forest como el modelo de mejor desempeño, se analiza la importancia de las variables utilizadas en la predicción. Este análisis permite identificar qué características tienen mayor peso en las decisiones del modelo y facilita la interpretación de los resultados desde una perspectiva de negocio.

In [ ]:
# Recuperar los nombres de las variables después del preprocesamiento

preprocesador_rf = modelo_rf.named_steps['preprocesamiento']

nombres_variables = preprocesador_rf.get_feature_names_out()

importancias = modelo_rf.named_steps['modelo'].feature_importances_

tabla_importancias = pd.DataFrame({'Variable': nombres_variables,'Importancia': importancias}).sort_values('Importancia', ascending=False)
display(tabla_importancias.head(15))

#grafico
top_importancias = tabla_importancias.head(15).sort_values('Importancia', ascending=True)

plt.figure(figsize=(8,6))

plt.barh(top_importancias['Variable'], top_importancias['Importancia'])

plt.xlabel('Importancia')
plt.ylabel('Variable')
plt.title('Importancia de variables - Random Forest')

plt.show()

El análisis de importancia de variables muestra que `loan_percent_income` es la variable con mayor peso relativo en las predicciones de Random Forest, seguida `por person_income` y `loan_amnt`. También presentan una contribución relevante `person_age`, `person_emp_length` y `cb_person_cred_hist_length`.

Entre las variables categóricas, la situación de vivienda, especialmente la categoría RENT, presenta una importancia mayor que las distintas categorías correspondientes al destino del préstamo (`loan_intent`) y al antecedente de incumplimiento (`cb_person_default_on_file`).

Estas importancias permiten identificar qué variables utiliza con mayor frecuencia el modelo para realizar sus predicciones, aunque no indican por sí mismas el sentido de la relación. Por ejemplo, que `person_income` tenga una importancia elevada no permite concluir a partir de este análisis que un mayor ingreso aumente o disminuya el riesgo de mora.

**5. Impacto en el negocio**

Una vez seleccionado Random Forest como modelo final, se analiza su impacto potencial desde una perspectiva económica. Para ello, se utiliza el monto solicitado (`loan_amnt`) como medida de exposición, comparando el monto correspondiente a los clientes que efectivamente entraron en mora con la parte de esa exposición que el modelo logra identificar anticipadamente.

Dado que el dataset no contiene información sobre recuperaciones posteriores al incumplimiento ni sobre el porcentaje efectivamente perdido por la entidad, los resultados deben interpretarse como exposición económica asociada a la mora y no como pérdida económica efectivamente evitada.

**5.1 Exposición económica asociada a la mora**

In [ ]:
# Crear una tabla de análisis económico sobre el conjunto de test
impacto_test = X_test[['loan_amnt']].copy()

impacto_test['loan_status_real'] = y_test
impacto_test['prediccion_rf'] = y_pred_rf_85

# Exposición total correspondiente a casos que realmente entraron en mora
exposicion_mora_total = impacto_test.loc[ impacto_test['loan_status_real'] == 1, 'loan_amnt'].sum()

# Exposición correspondiente a morosos detectados por el modelo
exposicion_mora_detectada = impacto_test.loc[(impacto_test['loan_status_real'] == 1) & (impacto_test['prediccion_rf'] == 1), 'loan_amnt'].sum()

# Exposición correspondiente a morosos no detectados
exposicion_mora_no_detectada = impacto_test.loc[(impacto_test['loan_status_real'] == 1) & (impacto_test['prediccion_rf'] == 0), 'loan_amnt'].sum()

porcentaje_exposicion_detectada = (exposicion_mora_detectada / exposicion_mora_total * 100)

print("Exposición total asociada a mora:", round(exposicion_mora_total, 2))
print("Exposición de mora detectada:", round(exposicion_mora_detectada, 2))
print("Exposición de mora no detectada:", round(exposicion_mora_no_detectada, 2))
print("Porcentaje de exposición en mora detectada:", round(porcentaje_exposicion_detectada, 2), "%")

En el conjunto de test, el monto total de los préstamos correspondientes a clientes que posteriormente entraron en mora asciende a 15.532.875. De esa exposición, Random Forest identifica anticipadamente 13.743.900, equivalentes al 88,48% del monto total asociado a mora. Por su parte, 1.788.975, equivalentes al 11,52%, corresponden a casos de mora que el modelo no logra detectar.

Resulta relevante que, si bien el modelo identifica el 85,9% de los clientes que entran en mora, esos casos representan el 88,48% de la exposición económica asociada a mora. Esto indica que los casos detectados concentran, en promedio, montos de préstamo algo mayores que los casos de mora no detectados.

Estos valores representan exposición económica identificada y no pérdida evitada. El dataset no contiene información sobre recuperaciones posteriores al incumplimiento ni sobre la proporción del préstamo que finalmente pierde la entidad, por lo que no sería correcto asumir que los 13.743.900 identificados constituyen un ahorro efectivo

**5.2 Escenarios de impacto económico**

Dado que el dataset no informa qué porcentaje del monto de un préstamo en mora se recupera posteriormente, no es posible calcular de manera directa la pérdida económica evitada por el modelo. Por este motivo, se plantean distintos escenarios hipotéticos de pérdida sobre la exposición correspondiente a los casos de mora detectados.

Estos escenarios no representan pérdidas reales observadas, sino que permiten dimensionar el potencial impacto económico del modelo bajo distintos supuestos.

In [ ]:
# Escenarios hipotéticos de pérdida sobre la exposición detectada

escenarios_perdida = [0.20, 0.50, 1.00]

resultados_escenarios = []

for porcentaje in escenarios_perdida:
    perdida_potencial = exposicion_mora_detectada * porcentaje

    resultados_escenarios.append({
        'Escenario de pérdida': f'{int(porcentaje * 100)}%',
        'Exposición de mora detectada': exposicion_mora_detectada,
        'Pérdida potencial asociada': perdida_potencial })

tabla_escenarios = pd.DataFrame(resultados_escenarios)

display(tabla_escenarios)

Bajo los escenarios hipotéticos considerados, la exposición correspondiente a los casos de mora detectados por el modelo podría asociarse a una pérdida potencial de 2.748.780 si se asumiera una pérdida del 20%, de 6.871.950 bajo un supuesto del 50% y de 13.743.900 en el escenario extremo de pérdida total.



**5.3 Beneficio neto: pérdida evitada versus el costo de rechazar buenos clientes**

El beneficio neto del modelo se mide comparando:

-*Lo que se evita perder*: cuando el modelo detecta clientes que efectivamente caerán en mora, la entidad se ahorra la pérdida que esos préstamos hubieran generado.
*   Esa pérdida se calcula con el LGD (Loss Given Default), que es el porcentaje del préstamo que realmente se pierde después de descontar intereses cobrados.
*  Fórmula: pérdida evitada = LGD × monto de los morosos detectados.

-*Lo que se deja de ganar*: cuando el modelo rechaza por error a clientes buenos (falsos positivos), la entidad pierde los intereses que esos préstamos habrían generado.
*   Se aproxima con el margen de interés promedio observado en el dataset (≈ 11%).
*   Fórmula: intereses perdidos = margen × monto de los falsos positivos.


El beneficio neto es entonces: Perdida evitada - Intereses perdidos

In [ ]:
MARGEN = 0.11      # tasa de interés promedio observada (11%): aproximación del ingreso por interés sobre el monto

monto = X_test['loan_amnt'].values
real = y_test.values
umbral_final = umbral_85_rf
y_prob_final = y_prob_rf
y_pred_final = y_pred_rf_85

def beneficio_neto(pred, lgd, margen=MARGEN):
    detectados = (pred == 1) & (real == 1)       #morosos detectados
    falsos_pos = (pred == 1) & (real == 0)       #buenos clientes marcados como riesgosos
    evitado = lgd * monto[detectados].sum()
    perdido = margen * monto[falsos_pos].sum()
    return evitado, perdido, evitado - perdido

filas = []
for lgd in [0.20, 0.50, 1.00]:
    evitado, perdido, neto = beneficio_neto(y_pred_final, lgd)
    filas.append({'LGD supuesta': f'{int(lgd*100)}%', 'Pérdida evitada': round(evitado),
                  'Intereses perdidos (FP)': round(perdido), 'Beneficio neto': round(neto),
                  'Neto por solicitud': round(neto / len(y_test))})
display(pd.DataFrame(filas).set_index('LGD supuesta'))

#LGD de equilibrio: por debajo de este valor, usar el modelo con este umbral no compensa
_, perdido, _ = beneficio_neto(y_pred_final, 0)
exp_detectada = monto[(y_pred_final == 1) & (real == 1)].sum()
print("LGD de equilibrio:", round(perdido / exp_detectada * 100, 1), "%")

En el caso del LGD 50% el modelo detecta 1.218 clientes que luego entran en mora, cuyos préstamos suman 13.743.900. Si de cada uno se pierde en promedio la mitad, rechazarlos evita una pérdida de 6.871.950. A cambio, el modelo marca como riesgosos a 1.781 clientes que sí habrían pagado, cuyos préstamos suman 16.809.075; al 11% de margen, rechazarlos implica renunciar a 1.848.998 en intereses. La diferencia, 5.022.952, es el beneficio neto estimado, es decir, unos 775 por solicitud evaluada.

El modelo conviene siempre que la LGD supere aproximadamente el 13,5% (LGD de equilibrio). Con una LGD del 20% el beneficio es mucho menor (899.782), porque el costo de los buenos clientes rechazados consume casi dos tercios de la pérdida evitada. Estos valores dependen de los supuestos de LGD y margen y no incluyen costos de fondeo, gastos operativos ni el efecto comercial de rechazar clientes, por lo que deben leerse como un orden de magnitud y no como un valor contable.

In [ ]:
# Beneficio neto según el umbral de decisión (es el Recall 85% el punto económicamente óptimo?)
from sklearn.metrics import recall_score, precision_score

umbrales = np.round(np.arange(0.05, 0.80, 0.01), 2)
plt.figure(figsize=(8, 5))
filas_opt = []
for lgd, color in [(0.20, '#4C72B0'), (0.50, '#55A868'), (1.00, '#C44E52')]:
    netos = [beneficio_neto((y_prob_final >= u).astype(int), lgd)[2] for u in umbrales]
    plt.plot(umbrales, np.array(netos) / 1e6, label=f'LGD {int(lgd*100)}%', color=color)
    i = int(np.argmax(netos)); pred_i = (y_prob_final >= umbrales[i]).astype(int)
    filas_opt.append({'LGD': f'{int(lgd*100)}%', 'Umbral óptimo': umbrales[i], 'Beneficio neto': round(netos[i]),
                      'Recall': round(recall_score(y_test, pred_i), 3), 'Precision': round(precision_score(y_test, pred_i), 3)})
plt.axvline(umbral_final, color='gray', linestyle='--', label=f'Umbral con Recall ≥ 85% ({umbral_final:.3f})')
plt.xlabel('Umbral de decisión'); plt.ylabel('Beneficio neto (millones)')
plt.title('Beneficio neto en test según umbral y supuesto de LGD'); plt.legend(); plt.show()
display(pd.DataFrame(filas_opt).set_index('LGD'))

El gráfico muestra que el umbral con Recall ≥ 85% (priorizar no dejar pasar morosos) no coincide siempre con el máximo beneficio económico:

*   Con LGD = 20%, el óptimo está en umbral ≈ 0,37 (Recall 62,7%, Precision 76,8%).
*   Con LGD = 50%, el óptimo está en ≈ 0,25 (Recall 73,0%).
*   Con LGD = 100%, coincide casi con el umbral adoptado.

Esto significa que el Recall mínimo del 85% es una decisión estratégica de apetito de riesgo, no necesariamente la más rentable. Con LGD = 50%, esa decisión cuesta unos 0,3 millones frente al óptimo económico.

**6. Limitaciones, riesgos y consideraciones éticas**

**6.1 Limitaciones del análisis**

El presente trabajo presenta algunas limitaciones que deben considerarse al interpretar los resultados. En primer lugar, se utiliza un dataset público con fines educativos, por lo que los datos no necesariamente representan las características de la cartera crediticia de una entidad financiera real.

Además, el dataset no dispone de un identificador único de cliente, lo que impide confirmar si los registros con valores idénticos correspondían efectivamente a clientes duplicados. Tampoco contiene información sobre recuperaciones posteriores a una mora, por lo que no es posible determinar la pérdida económica efectiva asociada a cada incumplimiento.

Por último, loan_grade y loan_int_rate fueron excluidas del modelado debido a que, de acuerdo con el momento de predicción definido, podrían contener información generada durante una etapa posterior de la evaluación crediticia y producir data leakage.

**6.2 Riesgos del modelo**

El modelo presenta también limitaciones propias de su capacidad predictiva. Random Forest alcanza un Recall de 85,9%, por lo que todavía existe un 14,1% de los casos de mora que no son identificados. Asimismo, la Precision de 40,6% implica que una proporción relevante de los clientes clasificados como riesgosos finalmente no entra en mora.

Por este motivo, las predicciones no deberían interpretarse como certezas, sino como estimaciones de riesgo. El umbral de decisión elegido responde al objetivo de priorizar la detección de posibles casos de mora, pero genera como contrapartida un aumento de los falsos positivos.

**6.3 Consideraciones éticas**

En un contexto real, la utilización de un modelo de riesgo crediticio puede tener consecuencias sobre el acceso de las personas al crédito. Por este motivo, las predicciones deberían utilizarse como herramienta de apoyo dentro del proceso de evaluación y no interpretarse de manera aislada como una decisión automática.

También sería necesario monitorear periódicamente el desempeño del modelo y analizar posibles diferencias en sus resultados entre distintos grupos de clientes, con el objetivo de detectar sesgos o comportamientos no deseados. Asimismo, debería garantizarse un uso adecuado y seguro de la información utilizada para realizar las predicciones.

Finalmente, las variables consideradas por el modelo deberían responder a criterios vinculados con el riesgo crediticio y su utilización debería ser transparente y justificable dentro del proceso de decisión



**7. Conclusiones y recomendaciones**

**7.1 Conclusiones**

El trabajo permitió desarrollar y evaluar modelos de clasificación orientados a identificar, durante la evaluación inicial de una solicitud de crédito, clientes con mayor riesgo de incurrir en mora.

Durante la preparación de los datos se realizó la limpieza de registros inconsistentes y duplicados, el tratamiento de valores faltantes y la transformación de variables numéricas y categóricas. Asimismo, se decidió excluir `loan_grade` y `loan_int_rate` del conjunto de variables predictoras, con el objetivo de evitar una posible fuga de información, considerando que la predicción se realiza antes de la asignación de las condiciones definitivas del préstamo.

Se evaluaron dos algoritmos, Regresión Logística y Random Forest, utilizando una división 80/20 entre entrenamiento y test y validación cruzada estratificada de 5 particiones sobre el conjunto de entrenamiento. Debido a la importancia otorgada a la detección de clientes que posteriormente incurren en mora, se analizaron diferentes umbrales de clasificación con el objetivo de alcanzar un Recall mínimo del 85%.

Random Forest presentó el mejor desempeño general. Con el umbral seleccionado de 0,123 alcanzó en el conjunto de test un Recall de 85,9%, Precision de 40,6%, F1-score de 0,552, ROC-AUC de 0,877 y PR-AUC de 0,785. Estos resultados fueron superiores a los obtenidos mediante Regresión Logística, especialmente en Precision, ROC-AUC y PR-AUC.

Desde el punto de vista económico, el modelo logró identificar el 88,48% de la exposición correspondiente a préstamos que posteriormente entraron en mora, equivalente a 13.743.900 sobre una exposición total de 15.532.875. Este resultado no representa una pérdida efectivamente evitada, sino el monto de exposición asociado a casos de mora que el modelo consigue identificar anticipadamente.

**7.2 Recomendaciones**

A partir de los resultados obtenidos, Random Forest podría utilizarse como herramienta de apoyo para la evaluación del riesgo crediticio, permitiendo identificar solicitudes que requieren un análisis adicional. La clasificación como cliente de mayor riesgo no debería implicar necesariamente el rechazo automático del crédito, sino que podría dar lugar a medidas alternativas de gestión de riesgo.

También sería conveniente incorporar información sobre las pérdidas efectivamente producidas ante una mora y los niveles de recuperación. Esto permitiría pasar del análisis de exposición realizado en este trabajo a una estimación más precisa del impacto económico real del modelo.

Finalmente, debido al efecto que una evaluación crediticia puede tener sobre los solicitantes, se recomienda complementar las predicciones con criterios de negocio y controles que permitan supervisar posibles sesgos y mantener decisiones justificables y transparentes.